[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-12-llm-context.ipynb)


# Lecture 12: does context help an LLM translate a language it barely knows?

The goal of this assignment is to explore how providing in-context examples affects the
translation quality of a large language model (LLM) for a low-resource language of your choice:
you show it a few translated sentences in the prompt, then more, and measure whether its
translations improve.

By the end you will be able to:

1. prompt a decoder-only model to translate a low-resource language from a few example pairs,
   with no training;
2. measure how much random and nearest examples improve translation, and tell a real difference
   from the spread between random draws;
3. explain how examples in the prompt help or hurt when the model's weights never change, and
   what sentence-by-sentence translation cannot fix.

These serve Lecture 12's outcome, which is these three together.

You write the two functions that are the method, marked TODO, and a prediction; checks tell you
when each is right, and the notebook runs the rest. It writes everything Assignment 12 asks you
to upload to `MyDrive/CS479/assignment12`; you upload this notebook too, once it has run.

**Running in Google Colab?** Runtime → Change runtime type → a GPU, then Run all; it stops at
each TODO and at the prediction until you write it. It runs on a built-in English–Spanish sample
until you change two lines in Step 1 for your language. When you finish, disconnect the GPU:
Runtime → Disconnect and delete runtime. The last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 12 (*Using LLMs for MT; Expanding Context Awareness*). A head start on **A12**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where this notebook saves what you upload.
from torchlingo.colab import course_dir, setup

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
OUT_DIR = course_dir(env.drive_dir, "assignment12", create=True)   # MyDrive/CS479/assignment12

import pandas as pd
import torch


## Step 1: your language's sentences, split into context and test

Two sets, kept apart. The **context pool** supplies the example pairs in the prompt; the **test
set** is what the model translates and what is scored. A sentence in both would let the model
copy its answer out of the prompt.

**Get your language's data.** Follow the data link on the Assignment 12 page in Learning Suite,
open `LLM_Assignment_Data`, then your language's folder (Kosraean's is spelled `Kosrean`), and
copy its two files, `src.txt` (English) and `tgt.txt` (your language), into your Drive as
`MyDrive/CS479/a12-data`. They are line-aligned: line 5 of one translates line 5 of the other.
Then set `LANGUAGE` and `DATA_DIR` in the cell below, together: with neither set, the notebook
runs on a built-in English–Spanish sample.

**Look at a few pairs before you trust them.** The data is real translation-memory text, and not
every line is translated: Efik's third line, for one, is the English left as it was. A pair like
that in the context pool shows the model an "example" that teaches it to copy English, and one in
the test set rewards copying. You may drop such pairs from `pairs` before the split; if you do,
say so in your write-up.

**Yours to write: `split_pool_and_test`**, two cells down. Its cell checks it on the alphabet's
26 pairs, whose split you can see, then runs the library's contamination check on your real split
before saving it.

In [ ]:
LANGUAGE = "Spanish"    # <- your A12 language: Efik, Kosraean, Palauan, Pohnpeian, Telugu or Yapese
DATA_DIR = None         # <- your language's folder, e.g. env.drive_dir / "CS479" / "a12-data"

SEED = 479
POOL_SIZE, TEST_SIZE = 500, 50   # 500 pairs to choose examples from, and 50 test sentences

In [ ]:
# Setup: run this cell, no need to read it. It loads your language's pairs, or the sample.
from pathlib import Path

if (DATA_DIR is None) != (LANGUAGE == "Spanish"):
    raise SystemExit(
        "Set LANGUAGE and DATA_DIR together: both for your A12 language, or neither for the "
        "built-in Spanish sample. Otherwise the prompt names one language and the data is another."
    )


def read_lines(path):
    """The file's lines. utf-8-sig drops a byte-order mark; reading as text turns Windows line
    endings into "\n"; splitting on "\n" alone keeps other line-break characters inside a line."""
    return path.read_text(encoding="utf-8-sig").rstrip("\n").split("\n")


def load_pairs(data_dir):
    """Return a DataFrame of English (src) and target-language (tgt) pairs from data_dir."""
    data_dir = Path(data_dir)
    missing = [name for name in ("src.txt", "tgt.txt") if not (data_dir / name).is_file()]
    if missing:
        found = sorted(p.name for p in data_dir.iterdir()) if data_dir.is_dir() else "no such folder"
        raise SystemExit(
            f"In {data_dir}: expected src.txt and tgt.txt from the A12 data folder; "
            f"missing {missing}, found {found}."
        )
    src, tgt = read_lines(data_dir / "src.txt"), read_lines(data_dir / "tgt.txt")
    if len(src) != len(tgt):
        raise SystemExit(f"src.txt has {len(src):,} lines and tgt.txt {len(tgt):,}; they must match.")
    frame = pd.DataFrame({"src": src, "tgt": tgt})
    frame = frame[(frame.src.str.strip() != "") & (frame.tgt.str.strip() != "")]
    return frame.drop_duplicates("src").reset_index(drop=True)


if DATA_DIR is None:
    from torchlingo.colab import fetch_data

    (sample,) = fetch_data(["data/example.tsv"])
    pairs = pd.read_csv(sample, sep="\t", dtype=str, keep_default_na=False)
    words = pairs.src.str.split().str.len()
    pairs = pairs[(pairs.kind == "transcript") & words.between(6, 20)][["src", "tgt"]]
    pairs = pairs.drop_duplicates("src").reset_index(drop=True)
    print("*** The built-in English-Spanish sample. Spanish is not the assignment: set LANGUAGE")
    print("*** and DATA_DIR above to your A12 language, then run all again. ***\n")
else:
    pairs = load_pairs(DATA_DIR)
assert len(pairs) >= POOL_SIZE + TEST_SIZE, f"only {len(pairs)} pairs; need {POOL_SIZE + TEST_SIZE}"
print(f"{len(pairs):,} {LANGUAGE} pairs loaded.")

In [ ]:
from torchlingo.diagnostics import check_contamination


def split_pool_and_test(pairs, pool_size, test_size, seed):
    """Return (pool, test): pool_size pairs to draw examples from, test_size pairs to translate.

    TODO: write it. Choose the pairs at random, the same ones every time for the same seed and
    different ones for a different seed. No sentence may be in both sets.
    """
    raise NotImplementedError("write split_pool_and_test, then run this cell again")


# A case whose answer you can check: the alphabet's 26 pairs into two and three.
toy = pd.DataFrame({"src": list("abcdefghijklmnopqrstuvwxyz"), "tgt": list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")})
toy_pool, toy_test = split_pool_and_test(toy, 2, 3, seed=0)
assert (len(toy_pool), len(toy_test)) == (2, 3), "the pool and test set are not the sizes asked for"
assert not set(toy_pool.src) & set(toy_test.src), "a pair is in both the pool and the test set"
assert split_pool_and_test(toy, 2, 3, seed=0)[1].equals(toy_test), "the same seed must give the same split"
assert not split_pool_and_test(toy, 2, 3, seed=1)[1].equals(toy_test), (
    "a different seed must give a different split: choose the pairs at random"
)

pool, test = split_pool_and_test(pairs, POOL_SIZE, TEST_SIZE, SEED)
result = check_contamination(test.src, pool.src, name="test")
print(result)
assert result, "a test sentence is also in the context pool, so the model could copy its answer"

test.to_csv(OUT_DIR / "test.tsv", sep="\t", index=False)
print(f"{len(pairs):,} {LANGUAGE} pairs; {len(pool)} in the context pool, {len(test)} in the test set.")
print(f"Saved {OUT_DIR / 'test.tsv'}")

## Step 2: the prompt

"Adding context" means putting *k* English sentences and their translations into the prompt,
above the sentence to translate. At *k* = 0 the model has only the instruction and whatever it
learned about your language in pretraining.

**Yours to write: `build_prompt`.** The checks confirm what it must contain; the wording is
yours. **Only the first line of the model's reply is kept as its translation**, so a reply that
starts "Sure, here is the translation:" scores as that sentence. Write the prompt so the reply
starts with the translation. The cell then prints your prompt at *k* = 5: everything the model
knows about the task.

In [ ]:
def build_prompt(sentence, examples):
    """Return the prompt asking for sentence's translation, with the example pairs above it.

    TODO: write it. examples is a list of (English, translation) pairs, possibly empty. The
    prompt must say what to do (translate into LANGUAGE), show every example pair, and end with
    the sentence to translate. The wording is yours: it changes the result, and the write-up
    asks about it. Asking for "the translation only" keeps the model from explaining itself.
    """
    raise NotImplementedError("write build_prompt, then run this cell again")


# Checks on a case you can read: two marked example pairs, and none.
two = build_prompt("the cat sleeps", [("good morning", "EXAMPLE-A"), ("thank you", "EXAMPLE-B")])
none = build_prompt("the cat sleeps", [])
assert LANGUAGE in none and "the cat sleeps" in none, "the prompt must name the language and the sentence"
for text in ("good morning", "EXAMPLE-A", "thank you", "EXAMPLE-B"):
    assert text in two, f"an example pair is missing from the prompt: {text!r}"
assert "EXAMPLE-A" not in none, "with no examples, the prompt must have none"
assert two.rindex("the cat sleeps") > two.rindex("EXAMPLE-B"), "the sentence must come after the examples"

examples = list(zip(pool.src, pool.tgt))
print(build_prompt(test.src.iloc[0], examples[:5]))


## Step 3: which examples

*Which* pairs go into the prompt can matter as much as how many. The notebook chooses them two
ways, from a pool of 500:

- **At random.** One random draw can be lucky or unlucky, so the notebook makes three, each with
  its own seed, and reports their mean and spread. Within one draw the 5 examples are part of the
  10 and the 10 part of the 20, so across sizes only how many the model sees changes.
- **Nearest.** For each test sentence, the *k* pool pairs whose English is most like it, the idea
  behind Moslem et al. (in the Lecture 12 deck). Every test sentence then has its own examples,
  chosen because they share its words. Moslem et al. measure "most like it" with sentence
  embeddings, the idea from A3; here it is the share of three-letter sequences two English
  sentences have in common, a cruder stand-in that needs no model.

The cell below shows both for the first test sentence.

In [ ]:
DRAW_SEEDS = [1, 2, 3]   # three random draws of the examples


def random_examples(seed, k):
    """k pool pairs at random. One seed is one order of the pool, so a draw's 5 are in its 10."""
    shuffled = pool.sample(frac=1, random_state=seed)
    return list(zip(shuffled.src, shuffled.tgt))[:k]


def trigrams(text):
    """The set of three-character sequences in text, lowercased, with a space at each end."""
    padded = f" {text.lower()} "
    return {padded[i:i + 3] for i in range(len(padded) - 2)}


pool_trigrams = [trigrams(s) for s in pool.src]


def nearest_examples(sentence, k):
    """The k pool pairs whose English shares the most trigrams with sentence, most similar last."""
    mine = trigrams(sentence)
    similarity = [len(mine & theirs) / len(mine | theirs) for theirs in pool_trigrams]
    ranked = sorted(range(len(pool)), key=lambda i: similarity[i], reverse=True)[:k]
    return [(pool.src.iloc[i], pool.tgt.iloc[i]) for i in reversed(ranked)]


sentence = test.src.iloc[0]
print(f"test sentence:  {sentence}\n")
for src, _ in reversed(nearest_examples(sentence, 3)):   # most similar first, for reading
    print(f"nearest:  {src}")
print()
for src, _ in random_examples(DRAW_SEEDS[0], 3):
    print(f"random:   {src}")

## Step 4: the model

The default is `Qwen/Qwen2.5-1.5B-Instruct`: a small, open chat model that needs no Hugging Face
token and fits a Colab GPU. You may choose another from the
[Hugging Face Hub](https://huggingface.co/models) that was not trained for translation (not
NLLB, M2M100 or MarianMT); name it in the write-up. Choose a chat (instruction-tuned) model,
since the prompt goes through its chat template; some, such as Llama and Gemma, are gated, and
need their license accepted on the model's page and a Hugging Face token. It decodes greedily, so
the same prompt always gives the same translation, and a change in score comes from the context
alone.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"     # <- or another chat model that is not MT-trained

tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16 if device.type == "cuda" else torch.float32
).to(device).eval()


def translate(sentences, examples_for, batch_size=8):
    """Translate sentences[i] with the pairs examples_for[i] in its prompt; greedy decoding."""
    out = []
    for start in range(0, len(sentences), batch_size):
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": build_prompt(s, examples)}],
                tokenize=False, add_generation_prompt=True,
            )
            for s, examples in zip(sentences[start:start + batch_size],
                                   examples_for[start:start + batch_size])
        ]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, max_new_tokens=128, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        for row in ids[:, batch["input_ids"].shape[1]:]:
            text = tokenizer.decode(row, skip_special_tokens=True).strip()
            out.append(text.splitlines()[0].strip() if text else "")
    return out


print(f"{MODEL} on {device}: {sum(p.numel() for p in model.parameters()):,} parameters")


## Step 5: predict, then translate at each context size

**Write a prediction first**, in `PREDICTION` below: which context size will score highest for
your language, whether the nearest examples will beat random ones, and why, in a sentence or two.
The notebook stops there until you write it; the write-up asks you to reconcile it with what
happens.

In [ ]:
PREDICTION = ""   # <- your prediction, in a sentence or two

assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
print("Prediction recorded.")

Now the model translates the same 50 test sentences thirteen times: once with no examples, then
at each of 5, 10 and 20 examples, three times with random draws and once with the nearest
examples. That is three times the translating of a single curve, and it is what turns one
lucky or unlucky draw into a result. Each run saves its translations as `hyp-*.txt`; the random
draws save their example pairs as `context-k-drawN.tsv`, and the nearest examples, one set per
test sentence, as `nearest-k.tsv`.

In [ ]:
from torchlingo.evaluation import compute_bleu, compute_chrf

CONTEXT_SIZES = [0, 5, 10, 20]
sources, references = list(test.src), list(test.tgt)
results = []


def record(condition, k, draw, hyps, name):
    """Save one run's translations as hyp-<name>.txt, and score them with BLEU and chrF."""
    (OUT_DIR / f"hyp-{name}.txt").write_text("\n".join(hyps) + "\n", encoding="utf-8")
    bleu, chrf = compute_bleu(hyps, references), compute_chrf(hyps, references)
    results.append({"condition": condition, "context": k, "draw": draw,
                    "bleu": round(bleu.score, 2), "chrf": round(chrf.score, 2),
                    "bleu_signature": str(bleu.signature), "chrf_signature": str(chrf.signature)})
    size = "" if k is None else f"k = {k:2}"
    print(f"{condition:<8} {size:<7} {'draw ' + str(draw) if draw else '':<7} "
          f"BLEU {bleu.score:6.2f}   chrF {chrf.score:6.2f}   e.g. {hyps[0]!r}")


record("none", 0, "", translate(sources, [[]] * len(sources)), "0")
for k in CONTEXT_SIZES[1:]:
    for draw, seed in enumerate(DRAW_SEEDS, start=1):
        chosen = random_examples(seed, k)
        pd.DataFrame(chosen, columns=["src", "tgt"]).to_csv(
            OUT_DIR / f"context-{k}-draw{draw}.tsv", sep="\t", index=False)
        record("random", k, draw, translate(sources, [chosen] * len(sources)), f"{k}-draw{draw}")

    nearest = [nearest_examples(s, k) for s in sources]
    pd.DataFrame([(s, src, tgt) for s, chosen in zip(sources, nearest) for src, tgt in chosen],
                 columns=["test_src", "src", "tgt"]).to_csv(
        OUT_DIR / f"nearest-{k}.tsv", sep="\t", index=False)
    record("nearest", k, "", translate(sources, nearest), f"nearest-{k}")

print(f"\nreference for that sentence: {references[0]!r}")

**Against a dedicated MT system.** Is an LLM with examples any match for a model trained to
translate? Where Meta's NLLB-200 covers your language, the next two cells translate the same 50
sentences with it, with no examples (it takes none), and greedily, as the LLM does, so the two
decode the same way. Of the A12 languages, NLLB covers only Telugu; for the others the cells
say so and add no row, and that absence is itself an answer to question 5.

In [ ]:
# Setup: run this cell, no need to read it. It loads NLLB, if it covers your language.
# The A12 languages among NLLB-200's 200, with its codes for them. Efik, Kosraean, Palauan,
# Pohnpeian and Yapese are not.
NLLB_CODES = {"Spanish": "spa_Latn", "Telugu": "tel_Telu"}
NLLB = "facebook/nllb-200-distilled-600M"
nllb_translate = None

if LANGUAGE in NLLB_CODES:
    from transformers import AutoModelForSeq2SeqLM

    nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB, src_lang="eng_Latn")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
        NLLB, dtype=torch.float16 if device.type == "cuda" else torch.float32
    ).to(device).eval()
    target = nllb_tokenizer.convert_tokens_to_ids(NLLB_CODES[LANGUAGE])

    def nllb_translate(sentences, batch_size=16):
        """Translate English sentences into LANGUAGE with NLLB; greedy, like the LLM."""
        out = []
        for start in range(0, len(sentences), batch_size):
            batch = nllb_tokenizer(sentences[start:start + batch_size], return_tensors="pt",
                                   padding=True).to(device)
            with torch.no_grad():
                ids = nllb_model.generate(**batch, forced_bos_token_id=target, num_beams=1,
                                          do_sample=False, max_new_tokens=128)
            out += nllb_tokenizer.batch_decode(ids, skip_special_tokens=True)
        return out

    print(f"{NLLB} loaded, translating into {NLLB_CODES[LANGUAGE]}.")

In [ ]:
if nllb_translate is None:
    print(f"NLLB-200 does not cover {LANGUAGE}, so there is no NLLB row. Say so in the write-up.")
else:
    record("NLLB", None, "", nllb_translate(sources), "nllb")

## Step 6: the report and the chart

Every run is a row of `scores.tsv`, with BLEU, chrF and both signatures. The table below puts the
three random draws at each size into one line: their mean, then the lowest and highest. Fifty
sentences is a small test set, and the spread between draws shows how much of a difference
between sizes is the luck of which examples were drawn. chrF is there because BLEU near zero
cannot separate anything: for a language the model barely knows, every size may score under a
few BLEU points, while chrF, which counts matching characters rather than whole words, still
moves.


In [ ]:
import importlib.metadata
import json

import matplotlib.pyplot as plt
import torchlingo

runs = pd.DataFrame(results)
runs["context"] = runs["context"].astype("Int64")   # whole numbers, and blank for NLLB's row
runs.insert(0, "model", MODEL)
runs.insert(0, "language", LANGUAGE)
runs.to_csv(OUT_DIR / "scores.tsv", sep="\t", index=False)

# The three random draws at each size become one row: their mean, and the lowest and highest.
summary = runs.groupby(["condition", "context"], sort=False, dropna=False).agg(
    bleu=("bleu", "mean"), bleu_low=("bleu", "min"), bleu_high=("bleu", "max"),
    chrf=("chrf", "mean"), chrf_low=("chrf", "min"), chrf_high=("chrf", "max"), runs=("bleu", "size"),
).reset_index()
for row in summary.itertuples():
    size = "" if pd.isna(row.context) else f"k = {int(row.context):2}"
    spread = (f"  (draws {row.bleu_low:.2f} to {row.bleu_high:.2f}; chrF {row.chrf_low:.2f} to "
              f"{row.chrf_high:.2f})" if row.runs > 1 else "")
    print(f"{row.condition:<8} {size:<7} BLEU {row.bleu:6.2f}   chrF {row.chrf:6.2f}{spread}")
print(f"BLEU signature: {results[0]['bleu_signature']}")
print(f"chrF signature: {results[0]['chrf_signature']}")
print(f"your prediction: {PREDICTION.strip()}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
start = summary[summary.condition == "none"]
for ax, metric in zip(axes, ("bleu", "chrf")):
    for condition, style in (("random", "o-"), ("nearest", "s--")):
        line = pd.concat([start, summary[summary.condition == condition]])
        errors = [line[metric] - line[f"{metric}_low"], line[f"{metric}_high"] - line[metric]]
        ax.errorbar(line.context, line[metric], yerr=errors, fmt=style, capsize=4,
                    label="random (mean, lowest to highest of 3 draws)" if condition == "random"
                    else "nearest examples")
    for row in summary[summary.condition == "NLLB"].itertuples():
        ax.axhline(getattr(row, metric), color="gray", linestyle=":", label="NLLB, no examples")
    ax.set_xticks(CONTEXT_SIZES)
    ax.set_xlabel("example pairs in the prompt")
    ax.set_ylabel("BLEU" if metric == "bleu" else "chrF")
axes[0].legend(fontsize=8)
fig.suptitle(f"{LANGUAGE}, {MODEL}")
fig.savefig(OUT_DIR / "scores-by-context.png", dpi=150, bbox_inches="tight")
plt.show()


def installed(package):
    """The installed version of package, or a note that it is not installed."""
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


# Everything someone needs to rerun this and get the same numbers.
rerun = {
    "language": LANGUAGE, "model": MODEL,
    "decoding": "greedy (do_sample=False), max_new_tokens=128, first line of the reply kept",
    "split_seed": SEED, "pool_size": POOL_SIZE, "test_size": TEST_SIZE,
    "draw_seeds": DRAW_SEEDS, "context_sizes": CONTEXT_SIZES,
    "nearest": "character-trigram overlap of the English, most similar example last",
    "nllb": NLLB if nllb_translate is not None else None,
    "torchlingo": torchlingo.version_label(), "transformers": installed("transformers"),
    "torch": torch.__version__, "sacrebleu": installed("sacrebleu"),
}
(OUT_DIR / "a12-rerun.json").write_text(json.dumps(rerun, indent=2) + "\n", encoding="utf-8")

print(f"\nUpload from {OUT_DIR}: test.tsv, the context-k-drawN.tsv and nearest-k.tsv files, "
      "scores.tsv, scores-by-context.png and a12-rerun.json, then this notebook (Step 7 says how).")

## Step 7: analysis

Answer these in your write-up, and say which model you used:

1. **What was the best context size for your language? Did you see a point where adding more
   context gave diminishing returns, or made translation worse?** Put your prediction from
   Step 5 beside the result, and say in a sentence what explains the difference, if any. Read
   the spread first: is the difference between two sizes larger than the difference between
   three draws at one size? If it is not, say what you can and cannot conclude.
2. **Why do you think in-context examples help (or hurt) translation quality? Explain the possible
   mechanisms.** The model's weights are the same at every size; say what "adding context"
   changes, then.
3. **Did the nearest examples beat random ones at the same size?** Find one test sentence whose
   translation they changed (compare `hyp-nearest-10.txt` with `hyp-10-draw1.txt`), look at its
   nearest examples in `nearest-10.tsv`, and say what they shared with it and whether the
   translation used it.
4. **Why must the context pool and the test set share no sentence?** Say what the score
   would have measured if they did.
5. **LLM against dedicated MT.** If NLLB covers your language, how does its row compare with the
   LLM's best? If it does not, say what that tells you about which languages get dedicated MT,
   and what the LLM offers a language that does not.
6. **What can no number of example pairs fix?** Each test sentence is translated alone, with
   example pairs above it but not the sentences around it in its document. Find a pronoun, or a
   name or term that appears more than once, in your test set's English, and say whether its
   translation changed between *k* = 0 and your best size. Then name two problems sentence-by-
   sentence translation leaves, such as a pronoun whose referent is in the sentence before, or a
   term translated two ways in one document, and for each say what the prompt would need to
   hold to fix it.
7. **What were the main challenges you met? (Model choice, prompt wording, compute, and so on.)**
8. **How would someone rerun this?** Paste the contents of `a12-rerun.json`: the model, the
   seeds, the decoding settings and the library versions.

**Upload** to your folder in the Assignment shared folder: `test.tsv`, the `context-k-drawN.tsv`
and `nearest-k.tsv` files, `scores.tsv`, the chart `scores-by-context.png`, `a12-rerun.json`,
the write-up, and **this notebook**: File → Download → Download .ipynb, after it has run, so every
cell's output is in it. The notebook shows your two functions, the prediction you wrote before
any score, and the real outputs, so the grader can check the write-up against it.

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. Everything this notebook writes is in `MyDrive/CS479/assignment12`, so disconnecting loses nothing.